# How To: Assess Image Quality Before Pipeline Design

Run diagnostics on a plate image to objectively assess noise, contrast,
and structure before choosing enhancers and detectors.

In [1]:
from phenotypic.data import load_yeast_plate
from phenotypic.plotting import PlotDiagnostics
from phenotypic.util import ImageMetricsCalculator
import matplotlib.pyplot as plt

In [2]:
plate = load_yeast_plate()
fig = PlotDiagnostics().inspect(plate)
fig

## Decision Guide

| Metric | Threshold | Action |
|--------|-----------|--------|
| Low SNR (< 10) | Noisy image | Add `StableDenoise` or `BlurGauss` |
| Low RMS contrast | Faint colonies | Add `CLAHE` or `ContrastStretching` |
| Low dynamic range | Under-exposed | Add `ContrastStretching` |
| Low gradient mean | Soft edges | Add `UnsharpMask` or `FocusEdgeSobel` |
| Long correlation length | Uneven illumination | Add `HomomorphicFilter` |

In [3]:
calculator = ImageMetricsCalculator(plate.detect_mat[:])
metrics = {
    "noise": calculator.compute_noise_metrics(),
    "contrast": calculator.compute_contrast_metrics(),
    "structure": calculator.compute_structure_metrics(),
    "background": calculator.compute_background_metrics(),
}
for category, values in metrics.items():
    print(f"\n{category}:")
    for key, val in values.items():
        if isinstance(val, (int, float)):
            print(f"  {key}: {val:.4f}")


noise:
  snr: 16.8730
  sigma_mad: 0.0197
  correlation_length: 49.5000

contrast:
  rms_contrast: 0.2567
  michelson: 0.4287
  dynamic_range: 0.0022
  p1: 0.2520
  p99: 0.6302

structure:
  mean_coherence: 0.2913
  optimal_scale: 1.0000
  peak_response: 0.0756

background:
  nonuniformity_ratio: 0.1346
  mean_gradient: 0.0005


In [4]:
plt.close("all")